In [1]:
#!pip install dspy pydantic

In [2]:
import dspy
from pydantic import BaseModel
from ftplib import FTP

In [3]:
test_accessions = [
    "GSE174188",
    "GSE209912",
    "GSE188367",
    "GSE136103"
]

# Tools

In [4]:
#!pip install scanpy

In [5]:
import re
import warnings
import tempfile
import os
import tarfile
import scanpy as sc
import gzip
import shutil
import pandas as pd
import anndata as ad

def get_geo_ftp_path(accession: str) -> str:
    """
    Return the FTP directory for a GEO accession (GSE or GSM).
    """
    prefix = accession[:3]     # GSE or GSM
    number = accession[3:]
    chunk = prefix + number[:-3] + "nnn"

    # the ftp site stores series and samples in directories named by the accession number with the last three digits replaced by 'nnn'
    if prefix == "GSE":
        return f"/geo/series/{chunk}/{accession}/suppl/"
    elif prefix == "GSM":
        return f"/geo/samples/{chunk}/{accession}/suppl/"
    else:
        raise ValueError("Only GSE or GSM supported")

def list_geo_files(accession: str):

    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()

    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    files = ftp.nlst()
    ftp.quit()
    return files

def download_geo_supp_file(accession: str, file_name:str, output_dir: str):
    ftp = FTP("ftp.ncbi.nlm.nih.gov")
    ftp.login()
    
    path = get_geo_ftp_path(accession)
    try:
        ftp.cwd(path)
    except:
        try:
            path = re.sub(r"suppl/$", "", path)
            ftp.cwd(path)
            warnings.warn(f"No supplementary files for: {accession}")
        except:
            raise FileNotFoundError(f"Could not find FTP path: {path}")

    local_file_path = os.path.join(output_dir, file_name)
    with open(local_file_path, "wb") as f:
        try:
            ftp.retrbinary(f"RETR {file_name}", f.write)
        except Exception as e:
            ftp.quit()
            raise e
    ftp.quit()
    return local_file_path

def list_tar_contents(file_name: str):
     with tarfile.open(file_name, "r:*") as tar:
        for member in tar.getmembers():
            print(member.name)

def unpack_tar_file(tar_file_path: str, output_dir: str):
    with tarfile.open(tar_file_path, "r") as tar:
        tar.extractall(path=output_dir)

def build_anndata(counts_directory: str, sample_name: str, outdir: str):
    adata = sc.read_10x_mtx(counts_directory)
    
    # add sample name to obs and store anndata in dictionary
    adata.obs["sample_name"] = sample_name
    # adatas[sample_name] = adata
    
    # make a subdirectory to store anndata files
    adata_dir = os.path.join(outdir, "adatas")
    os.makedirs(adata_dir, exist_ok=True)
    adata.write_h5ad(os.path.join(adata_dir, f"{sample_name}.h5ad"))

    # check that the file was actually saved
    saved_file_path = os.path.join(adata_dir, f"{sample_name}.h5ad")
    if os.path.exists(saved_file_path):
        print(f"Anndata object successfully saved at: {saved_file_path}")
    else:
        raise FileNotFoundError(f"Failed to save anndata object at: {saved_file_path}")
    
    return saved_file_path

# Rename files according to 10x Genomics conventions
def rename_geo_files(directory: str):
    files = os.listdir(directory)
    
    matrix = None
    features = None
    barcodes = None

    for f in files:
        n = f.lower()

        # matrix
        if "mtx" in n:
            matrix = f
            continue

        # features (genes)
        if any(x in n for x in ["gene", "feature", "symbol"]):
            features = f
            continue

        # barcodes (cells)
        if any(x in n for x in ["barcode", "cell"]):
            barcodes = f
            continue

    # Safety check
    if not (matrix and features and barcodes):
        raise ValueError(
            f"Could not find all required files in {directory}. "
            f"Found matrix={matrix}, features={features}, barcodes={barcodes}"
        )

    rename_map = {
        matrix: "matrix.mtx",
        features: "features.tsv",
        barcodes: "barcodes.tsv"
    }

    # check if files are gzipped and add appropriate extension to the new name
    for key, value in list(rename_map.items()):
        if key.endswith(".gz"):
            rename_map[key] = value + ".gz"

    # apply the new names by using a bash mv command
    for old, new in rename_map.items():
        src = os.path.join(directory, old)
        dst = os.path.join(directory, new)
        shutil.move(src, dst)
        print(f"Renamed {src} → {dst}")

    return directory, rename_map

# structure 10x directory
def structure_10x_directory(directory):
    # create a new directory for the 10x files
    counts_directory = os.path.join(directory, "10x_counts")
    os.makedirs(counts_directory, exist_ok=True)

    # move the relevant files to the new directory
    for file in os.listdir(directory):
        if file in ["matrix.mtx", "matrix.mtx.gz", "features.tsv", "features.tsv.gz", "barcodes.tsv", "barcodes.tsv.gz"]:
            src = os.path.join(directory, file)
            dst = os.path.join(counts_directory, file)
            shutil.move(src, dst)
    
    return counts_directory

# convert csv files to tsv files
def convert_csv_to_tsv(file_path):
        
    # handle uncompressed csvs
    if file_path.endswith(".csv"):
        tsv_file_path = file_path[:-4] + ".tsv" # change suffix
        with open(file_path, "r") as csv_file, open(tsv_file_path, "w") as tsv_file:
            for line in csv_file:
                tsv_file.write(line.replace(",", "\t"))

    # handle gzipped csvs
    if file_path.endswith(".csv.gz"):
        tsv_file_path = file_path[:-7] + ".tsv.gz" # change suffix
        with gzip.open(file_path, "rt") as csv_file, gzip.open(tsv_file_path, "wt") as tsv_file:
            for line in csv_file:
                tsv_file.write(line.replace(",", "\t"))

    if not file_path.endswith(".csv.gz") and not file_path.endswith(".csv"):
        tsv_file_path = file_path  # no conversion needed
    return tsv_file_path

# a simple function to get the dimensions of the counts matrix to help with reformatting input files
def get_matrix_dimensions(matrix_file_path: str) -> tuple:
    # get dimensions of sparse matrix
    opener = gzip.open if matrix_file_path.endswith(".gz") else open

    with opener(matrix_file_path, "rt") as f:
        for line in f:
            line = line.strip()
            # skip comments and header
            if line.startswith("%") or line.startswith("%%"):
                continue

            # first non-comment line should be: rows cols nnz
            parts = line.split()
            if len(parts) == 3:
                rows, cols, nnz = map(int, parts)
                return rows, cols, nnz
    raise ValueError("Could not determine matrix dimensions from file.")

# check the formatting of the features file and reformat if necessary.
# likely problems are only a single column or a header row when there shouldn't be one.
def format_features_file(features_file_path: str, matrix_dimensions: tuple):

    # if file contains only one column, add a second column identical to the first with name gene_id
    if features_file_path.endswith(".gz"):
        with gzip.open(features_file_path, "rt") as f:
            lines = f.readlines()
    else:
        with open(features_file_path, "r") as f:
            lines = f.readlines()
    
    # check if there is exactly one more row than the number of rows in the matrix (indicating a header)
    nrows = len(lines)
    if nrows == matrix_dimensions[0] + 1:
        header = True
    else:
        header = False
    
    # if there's a header, parse accordingly
    if header:
        header = lines[0].strip().split("\t")
        rows = [line.strip().split("\t") for line in lines[1:]]
    else:
        rows = [line.strip().split("\t") for line in lines]

    # read lines as a pandas dataframe to check number of columns
    features_df = pd.DataFrame(rows)
    # if only one column, duplicate it as column 2 and add dummy column 3
    if features_df.shape[1] == 1:
        features_df[1] = features_df.iloc[:, 0]
        features_df[2] = "gene"
    elif features_df.shape[1] == 2:
        features_df[2] = "gene"
    # keep only first three columns if more are present
    features_df = features_df.iloc[:, :3]
    
    # write the reformatted features file back to disk
    if features_file_path.endswith(".gz"):
        with gzip.open(features_file_path, "wt") as f:
            features_df.to_csv(f, sep="\t", index=False, header = False)
    else:
        with open(features_file_path, "w") as f:
                features_df.to_csv(f, sep="\t", index=False, header = False)

    return features_file_path

def format_barcodes_file(barcodes_file_path: str, matrix_dimensions: tuple):

    # read barcodes file
    if barcodes_file_path.endswith(".gz"):
        with gzip.open(barcodes_file_path, "rt") as f:
            lines = f.readlines()
    else:
        with open(barcodes_file_path, "r") as f:
            lines = f.readlines()
    
    # check if there is exactly one more row than the number of columns in the matrix (indicating a header)
    nrows = len(lines)
    if nrows == matrix_dimensions[1] + 1:
        header = True
    else:
        header = False
    
    # if there's a header, parse accordingly
    if header:
        header = lines[0].strip().split("\t")
        rows = [line.strip().split("\t") for line in lines[1:]]
    else:
        rows = [line.strip().split("\t") for line in lines]
        

    # read lines as a pandas dataframe to check number of columns
    barcodes_df = pd.DataFrame(rows)
    # keep only first column if more are present
    barcodes_df = barcodes_df.iloc[:, :1]
    
     # strip any suffix
    tenx_pattern = r"([ACGTN]{16,20}-\d+)"
    barcodes_df[barcodes_df.columns[0]] = barcodes_df[barcodes_df.columns[0]].str.extract(tenx_pattern)

    # write the reformatted barcodes file back to disk
    if barcodes_file_path.endswith(".gz"):
        with gzip.open(barcodes_file_path, "wt") as f:
            barcodes_df.to_csv(f, sep="\t", index=False, header = False)
    else:
        with open(barcodes_file_path, "w") as f:
                barcodes_df.to_csv(f, sep="\t", index=False, header = False)

    return barcodes_file_path
    


In [6]:
tmpdir = tempfile.mkdtemp()
file_lists = {acc: list_geo_files(acc) for acc in test_accessions}
file_lists

/var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/ipykernel_50261/2190850001.py:40: UserWarning: No supplementary files for: GSE174188
  warnings.warn(f"No supplementary files for: {accession}")


{'GSE174188': ['matrix', 'miniml', 'soft'],
 'GSE209912': ['GSE209912_counts.mtx.gz',
  'GSE209912_readme.xls',
  'GSE209912_barcodes.csv.gz',
  'GSE209912_metadata.csv.gz',
  'GSE209912_symbols.csv.gz'],
 'GSE188367': ['GSE188367_atac_tf_counts.tar.gz',
  'filelist.txt',
  'GSE188367_RAW.tar'],
 'GSE136103': ['filelist.txt', 'GSE136103_RAW.tar']}

In [ ]:
tar_file = "GSE188367_RAW.tar"

accession = "GSE188367"

download_geo_supp_file(accession, tar_file, tmpdir)
list_tar_contents(tmpdir + "/" + tar_file)

In [ ]:
unpack_tar_file(tmpdir + "/" + tar_file, tmpdir)

list_tar_contents(tmpdir + "/" + "GSM5678317_BM-Old4_counts.tar.gz")

/var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/ipykernel_50244/2190850001.py:80: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=output_dir)


BM-Old4_counts
BM-Old4_counts/barcodes.tsv.gz
BM-Old4_counts/matrix.mtx.gz
BM-Old4_counts/features.tsv.gz


# Pydantic data classes

In [ ]:
class GEO_entry(BaseModel):
    accession: str
    title: str
    summary: str
    overall_design: str
    contributor: str
    pubmed_ids: list
    supplementary_files: list

class expression_data(BaseModel):
    sample_id: str
    gene_ids: list
    counts: list
    accession: str

class anndata_object(BaseModel):
    adata: object
    accession: str

# DSPy Agents

In [ ]:
class DSPyGEOFetcher(dspy.Signature):
    """You are a computational biologist that goes through NCBI GEO uploads and fetches expression data and turns it into an anndata object.
    
    You are given an accession number and a set of tools. 
    You will go and retrieve data. 
    Sometimes it will be in tar files. 

    Then you will process the data files, converting and renaming as necessary. Then process into an anndata object, save it, and report the location.
    
    You must decide which tools are the best to handle the user's request."""

    user_request: str = dspy.InputField()
    directory: str = dspy.InputField(
        desc = (
                    "The temporary directory where all downloaded and processed files should be stored."
                )
    )
    accession: str = dspy.InputField(
        desc = (
                    "The GEO accession number (GSE or GSM) to fetch data for."
                )
    )
    process_result: str = dspy.OutputField(
        desc = (
                    "Message that summarizes the process result and the information retrieved."
                )
    )
        
    

In [ ]:
agent = dspy.ReAct(
    DSPyGEOFetcher,
    tools = [
        get_geo_ftp_path,
        list_geo_files,
        list_tar_contents,
        unpack_tar_file,
        download_geo_supp_file,
        convert_csv_to_tsv,
        rename_geo_files,
        structure_10x_directory,
        get_matrix_dimensions,
        format_features_file,
        format_barcodes_file,
        build_anndata

    ],
    max_iters=20
)

In [ ]:
import sys
import os

In [ ]:
# get the Claude API key from local text file
# check if we're on MacOS or Windows and read appropriate file
if sys.platform.startswith("win"):
    with open("C:/Users/David/.claude_api.txt") as f:
        claude_key = f.read().strip()
else:
    with open("/Users/tatarakis/.api-keys/tatarakis-test-key.txt") as f:
        claude_key = f.read().strip()


In [ ]:
# define the language model to be used by all dspy agents in this notebook
lm = dspy.LM('anthropic/claude-sonnet-4-5-20250929', api_key=claude_key)
dspy.configure(lm=lm)

In [ ]:
# test_message = lm(messages=[{"role": "user", "content": "Confirm that we're ready to use Claude for agents. But do it as if you are Columbo when he suspects the user is the murderer but is still trying to be coy and disarming."}])  # => ['This is a test!']

# print(test_message[0])

In [ ]:
test_accession = test_accessions[1]

In [ ]:
prediction = agent(user_request = "oh mighty agent, what is your purpose? Answer me like Columbo.")

2026/01/03 12:02:45 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['user_request', 'trajectory']. Missing: ['directory', 'accession'].
2026/01/03 12:02:45 WARNING dspy.predict.predict: Not all input fields were provided to module. Present: ['user_request', 'trajectory']. Missing: ['directory', 'accession'].


In [ ]:
print(list(prediction.trajectory.values())[0])

Well, you know, that's a funny thing you ask. See, I'm just a humble computational biologist agent - nothing fancy. My purpose? Oh, it's pretty straightforward, really. I fetch data from NCBI GEO - that's the Gene Expression Omnibus, if you're not familiar. I take these accession numbers, right? GSE this, GSM that. And I go hunting through their FTP servers, downloading files, unpacking tar archives when I need to. Sometimes they're messy, sometimes they're neat. 

Then - and this is the interesting part - I process all that expression data. Convert formats here, rename files there, structure the directories just so. And at the end of it all? I build myself a nice anndata object. You know, for the scientists. Save it somewhere they can use it.

But here's the thing that's bothering me about your request - and maybe it's nothing, probably is nothing - but you're asking me about my purpose in a philosophical way, when really, I should be processing some GEO accession data. There's no acc

In [ ]:
tmpdir = tempfile.mkdtemp()

In [ ]:
os.listdir(tmpdir)

[]

In [21]:
adatas = {}

In [22]:
agent(user_request = (
    "I need you to download and process the data for accession " + 
    test_accession + 
    ". Take the following steps: " +
    "1) list and download the supplemental files for that accession" +
    "2) if there's a tar file, unpack it. If there are more tar files, do this for them as well. " +
    "3) look for 10X files matrix, features, and barcodes. If there are any .csv or .csv.gz files, turn them into .tsv/.tsv.gz files."
    "4) rename the files to match the 10X specifications for scanpy's read_10x function. Save the prefix stripped off as sample_name if there is one" + 
    "5) call structure_10x_directory(directory) and place the new folder inside a folder named based on the sample_name" +
    "6) take the returned counts_directory" +
    "7) call build_anndata(counts_directory, sample_name)" +
    "8) if that fails, try to format the features file by adding a second column identical to the first with name gene_id, then " +
    "9) call build_anndata(counts_directory, sample_name)" +
    "After organizing the files, you MUST call structure_10x_directory(directory). Then you MUST call build_anndata(counts_directory, sample_name, directory) using the returned counts_directory as input and the top directory as the outdir. Do not stop until these tools have been executed. Return the output path of build_anndata."
    ), 
    directory = tmpdir,
    accession = test_accession
)

Renamed /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/GSE209912_counts.mtx.gz → /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/matrix.mtx.gz
Renamed /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/GSE209912_symbols.tsv.gz → /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/features.tsv.gz
Renamed /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/GSE209912_barcodes.csv.gz → /var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmp9sk0hf1j/barcodes.tsv.gz


/Users/tatarakis/DSPy_agent_test/.venv_3.13/lib/python3.13/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/Users/tatarakis/DSPy_agent_test/.venv_3.13/lib/python3.13/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Prediction(
    trajectory={'thought_0': 'I need to start by listing the supplemental files available for accession GSE209912. This will help me understand what files are available and whether there are tar files that need to be unpacked. The first step is to use the list_geo_files tool to see what supplemental files are available for this accession.', 'tool_name_0': 'list_geo_files', 'tool_args_0': {'accession': 'GSE209912'}, 'observation_0': ['GSE209912_counts.mtx.gz', 'GSE209912_readme.xls', 'GSE209912_barcodes.csv.gz', 'GSE209912_metadata.csv.gz', 'GSE209912_symbols.csv.gz'], 'thought_1': 'Good! I can see the supplemental files for GSE209912. There are no tar files to unpack. I can see:\n- GSE209912_counts.mtx.gz (the matrix file)\n- GSE209912_barcodes.csv.gz (barcodes, but in CSV format - needs conversion to TSV)\n- GSE209912_symbols.csv.gz (likely the features/genes file, also in CSV format - needs conversion to TSV)\n- GSE209912_metadata.csv.gz (metadata)\n- GSE209912_readme.xls

In [27]:
os.listdir(os.path.join(tmpdir))

['GSE209912_barcodes.tsv.gz',
 'GSE209912_symbols.csv.gz',
 'adatas',
 '10x_counts']

{}

In [52]:
# testing reformatting functions

matrix_file_path = os.path.join(tmpdir, "10x_counts", "matrix.mtx.gz")
features_file_path = os.path.join(tmpdir, "10x_counts", "features.tsv.gz")
barcodes_file_path = os.path.join(tmpdir, "10x_counts", "barcodes.tsv.gz")

dims = get_matrix_dimensions(matrix_file_path)
dims

(19737, 74758, 260167518)

In [ ]:
sc.read_10x_mtx(os.path.join(tmpdir, "10x_counts"))

In [49]:
format_features_file(features_file_path=features_file_path, matrix_dimensions=dims)

'/var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmpdtjyn8fa/10x_counts/features.tsv.gz'

In [81]:
with gzip.open(features_file_path, "rt") as f:
    for i in range(5):
        print(f.readline().strip())

Sox17	Sox17	gene
Mrpl15	Mrpl15	gene
Lypla1	Lypla1	gene
Tcea1	Tcea1	gene
Atp6v1h	Atp6v1h	gene


In [68]:
format_barcodes_file(barcodes_file_path=barcodes_file_path, matrix_dimensions=dims)

'/var/folders/4q/7zxjlbf95fdd931ljqdvs9300000gn/T/tmpdtjyn8fa/10x_counts/barcodes.tsv.gz'

In [69]:
with gzip.open(barcodes_file_path, "rt") as f:
    for i in range(5):
        print(f.readline().strip())

AAACCCAAGCATACTC-1
AAACCCAAGTACAGAT-1
AAACCCACACTCGATA-1
AAACCCACACTGGCCA-1
AAACCCAGTACCACGC-1


In [76]:
test_adata = sc.read_10x_mtx(os.path.join(tmpdir, "10x_counts"), var_names="gene_symbols", make_unique=True)

/Users/tatarakis/DSPy_agent_test/.venv_3.13/lib/python3.13/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [77]:
test_adata

AnnData object with n_obs × n_vars = 74758 × 0
    var: 'gene_ids', 'feature_types'

In [ ]:
test_adata.var

Index([], dtype='object')

In [47]:
os.listdir(tmpdir + "/10x_counts")

['features.tsv.gz', 'barcodes.tsv.gz', 'matrix.mtx.gz']

In [73]:
dims

(19737, 74758, 260167518)

In [ ]:
tmpdir

In [ ]:
adatas

In [ ]:
rename_geo_files(directory=tmpdir, accession="GSE209912")

In [ ]:
os.listdir(tmpdir)